# 03 — Chargement en base DuckDB

Ce notebook charge les 14 CSV transformés (`data/processed/`) dans DuckDB via SQLAlchemy ORM + Alembic.

**Pipeline ETL :**
```
01_extract              Kaggle → data/raw/*.csv
02_transform            data/raw/*.csv → data/processed/*.csv (14 tables 3NF)
03_load (ici)           data/processed/*.csv → DuckDB (data/rl.duckdb)
```

## 1. Migrations Alembic

In [ ]:
from src.etl.load.core import apply_migrations

apply_migrations()
print('Schema a jour.')

## 2. Truncate (reload idempotent)

In [ ]:
from src.etl.load.core import truncate_all

truncated = truncate_all()
print(f'{len(truncated)} tables videes.')

## 3. Insertion via ORM

In [ ]:
from src.etl.load.core import insert_all

counts = insert_all()
for name, count in counts.items():
    print(f'  {name:<20} {count:>12,} lignes')

total = sum(counts.values())
print(f'\nTotal : {total:,} lignes')

## 4. Vérification SQL

In [ ]:
import pandas as pd
from src.database.engine import engine

query = """
SELECT
    'country' AS table_name, COUNT(*) AS rows FROM country
UNION ALL SELECT 'region', COUNT(*) FROM region
UNION ALL SELECT 'map', COUNT(*) FROM map
UNION ALL SELECT 'car', COUNT(*) FROM car
UNION ALL SELECT 'player', COUNT(*) FROM player
UNION ALL SELECT 'team', COUNT(*) FROM team
UNION ALL SELECT 'event', COUNT(*) FROM event
UNION ALL SELECT 'stage', COUNT(*) FROM stage
UNION ALL SELECT 'match', COUNT(*) FROM match
UNION ALL SELECT 'game', COUNT(*) FROM game
UNION ALL SELECT 'game_player', COUNT(*) FROM game_player
UNION ALL SELECT 'game_team', COUNT(*) FROM game_team
UNION ALL SELECT 'stat_type', COUNT(*) FROM stat_type
UNION ALL SELECT 'stat', COUNT(*) FROM stat
"""

df_check = pd.read_sql(query, engine)
display(df_check)
print(f'\nTotal en base : {df_check["rows"].sum():,} lignes')

## 5. Aperçu des données

In [ ]:
print('--- Top 5 teams ---')
display(pd.read_sql('SELECT * FROM team LIMIT 5', engine))

print('\n--- Top 5 games ---')
display(pd.read_sql('SELECT * FROM game LIMIT 5', engine))

print('\n--- Stat categories ---')
display(pd.read_sql('SELECT category, COUNT(*) as n FROM stat_type GROUP BY category ORDER BY n DESC', engine))

In [ ]:
print('Chargement termine.')
print(f'Base : data/rl.duckdb')
print(f'{df_check["rows"].sum():,} lignes dans 14 tables')